# Text Prepocessing

## Anggota Kelompok
- Najla Zahira (25031554041)
- Moza Athaillah Azzahra (25031554185)

Dataset
https://drive.google.com/file/d/15VDyty0SYiUD50JVP3ueY455trwUO4ml/view?usp=sharing

## Step 1 - Load dan Verifikasi Dataset

In [9]:
import re
import pandas as pd

df = pd.read_csv('hasil_scraping_api.csv')

print(f'Total Artikel: {len(df)}')
df.head()

Total Artikel: 100


,no,judul,teks,topik
0,1,Algoritma bias,Algoritma bias menggambarkan galat sistematis ...,Computer Science
1,2,Algoritma Lanczos,Algoritma Lanczos adalah algoritma iteratif ad...,Computer Science
2,3,ASCII,(ASCII) singkatan dari American Standard Code ...,Computer Science
3,4,Karakter null,"Karakter null (atau bita null), disingkat NUL ...",Computer Science
4,5,Komputasi optis,Komputasi optis adalah penggunaan foton yang d...,Computer Science


## Step 2 - Pembersihan Latex dan Math

In [10]:
def clean_latex_and_math(text):
  if not isinstance(text, str):
    return ''

  text = re.sub(r'\{\\displaystyle[\s\S]*?\}', ' ', text)

  latex_commands = r'\\(frac|left|right|mathbb|mathbbm|mathrm|mathbf|text|begin|end|in|times|cdot|to|mapsto|alpha|beta|gamma|delta|lambda|sigma|omega|infty|sum|prod|int)'
  text = re.sub(latex_commands, ' ', text)

  text = re.sub(r'\\', ' ', text)

  return text

df['teks_step2'] = df['teks'].apply(clean_latex_and_math)

print('Pembersihan LaTeX selesai!')

Pembersihan LaTeX selesai!


## Step 3 - Pembersihan Special-Character & Invalid Unicode Normalization

In [11]:
def clean_unicode_and_specials(text):
  if not isinstance(text, str):
    return ''

  text = re.sub(r'[\uE000-\uF8FF]', ' ', text)

  text = re.sub(r'\[\d+\]', ' ', text)

  text = re.sub(r'[^\w\s\+\#\-\.]', ' ', text)

  return text

df['teks_step3'] = df['teks_step2'].apply(clean_unicode_and_specials)

pua_remains = df['teks_step3'].apply(
    lambda x: len(re.findall(r'[\uE000-\uF8FF]', str(x)))
).sum()
print(f'Pembersihan Step 3 Selesai!')
print(f'Sisa Karakter Unicode PUA: {pua_remains}')

df[['judul', 'teks', 'teks_step2', 'teks_step3']].head()

Pembersihan Step 3 Selesai!
Sisa Karakter Unicode PUA: 0


,judul,teks,teks_step2,teks_step3
0,Algoritma bias,Algoritma bias menggambarkan galat sistematis ...,Algoritma bias menggambarkan galat sistematis ...,Algoritma bias menggambarkan galat sistematis ...
1,Algoritma Lanczos,Algoritma Lanczos adalah algoritma iteratif ad...,Algoritma Lanczos adalah algoritma iteratif ad...,Algoritma Lanczos adalah algoritma iteratif ad...
2,ASCII,(ASCII) singkatan dari American Standard Code ...,(ASCII) singkatan dari American Standard Code ...,ASCII singkatan dari American Standard Code ...
3,Karakter null,"Karakter null (atau bita null), disingkat NUL ...","Karakter null (atau bita null), disingkat NUL ...",Karakter null atau bita null disingkat NUL ...
4,Komputasi optis,Komputasi optis adalah penggunaan foton yang d...,Komputasi optis adalah penggunaan foton yang d...,Komputasi optis adalah penggunaan foton yang d...


## Step 4 - Pembersihan URL

In [12]:
def clean_urls(text):
  if not isinstance(text, str):
    return ''

  text = re.sub(r'https?://[^\s]+|www\.[^\s]+', ' ', text)

  return text

df['teks_step4'] = df['teks_step3'].apply(clean_urls)

url_remains = df['teks_step4'].apply(lambda x: len(re.findall(r'https?://|www\.', str(x)))).sum()
print('Pembersihan Step 4 Selesai!')
print(f'Sisa Tautan URL di Dataset: {url_remains}')

df[['judul', 'teks_step3', 'teks_step4']].head()

Pembersihan Step 4 Selesai!
Sisa Tautan URL di Dataset: 0


,judul,teks_step3,teks_step4
0,Algoritma bias,Algoritma bias menggambarkan galat sistematis ...,Algoritma bias menggambarkan galat sistematis ...
1,Algoritma Lanczos,Algoritma Lanczos adalah algoritma iteratif ad...,Algoritma Lanczos adalah algoritma iteratif ad...
2,ASCII,ASCII singkatan dari American Standard Code ...,ASCII singkatan dari American Standard Code ...
3,Karakter null,Karakter null atau bita null disingkat NUL ...,Karakter null atau bita null disingkat NUL ...
4,Komputasi optis,Komputasi optis adalah penggunaan foton yang d...,Komputasi optis adalah penggunaan foton yang d...


## Step 5 - Whitespace Normalization

In [13]:
def normalize_whitespace(text):
  if not isinstance(text, str):
    return ''

  text = re.sub(r'[\r\n\t]+', ' ', text)

  text = re.sub(r'\s+', ' ', text)

  return text.strip()

df['teks_step5'] = df['teks_step4'].apply(normalize_whitespace)

newline_remains = df['teks_step5'].apply(lambda x: len(re.findall(r'[\r\n\t]', str(x)))).sum()
multispace_remains = df['teks_step5'].apply(lambda x: len(re.findall(r'\s{2,}', str(x)))).sum()

print('Normalisasi Whitespace Step 5 Selesai!')
print(f'Sisa Enter/Tab (\\n, \\r, \\t): {newline_remains}')
print(f'Sisa Spasi Ganda Bertumpuk: {multispace_remains}')

print('\n--- SAMPEL SETELAH STEP 5 (Algoritma Lanczos) ---')
print(df[df['judul'] == 'Algoritma Lanczos']['teks_step5'].values[0][:300])

Normalisasi Whitespace Step 5 Selesai!
Sisa Enter/Tab (\n, \r, \t): 0
Sisa Spasi Ganda Bertumpuk: 0

--- SAMPEL SETELAH STEP 5 (Algoritma Lanczos) ---
Algoritma Lanczos adalah algoritma iteratif adaptasi dari metode daya power method untuk menemukan m nilai dan vektor eigen yang paling berguna umumnya yang tertinggi terendah dari sebuah matriks Hermite berukuran n n dengan m tidak perlu jauh lebih kecil dari n . Algoritma ini dirancang oleh Cornel


## Step 6 - Case Folding

In [14]:
def case_folding(text):
  if not isinstance(text, str):
    return ""

  return text.lower()

df["teks_step6"] = df["teks_step5"].apply(case_folding)

uppercase_remains = df["teks_step6"].apply(lambda x: len(re.findall(r"[A-Z]", str(x)))).sum()

print("Case Folding Step 6 Selesai!")
print(f"Sisa Huruf Kapital (A-Z) di Dataset: {uppercase_remains}")

print("\n--- SAMPEL SETELAH STEP 6 (Algoritma Lanczos) ---")
print(df[df["judul"] == "Algoritma Lanczos"]["teks_step6"].values[0][:250])

Case Folding Step 6 Selesai!
Sisa Huruf Kapital (A-Z) di Dataset: 0

--- SAMPEL SETELAH STEP 6 (Algoritma Lanczos) ---
algoritma lanczos adalah algoritma iteratif adaptasi dari metode daya power method untuk menemukan m nilai dan vektor eigen yang paling berguna umumnya yang tertinggi terendah dari sebuah matriks hermite berukuran n n dengan m tidak perlu jauh lebih 


## Step 7 - Tokenization

In [15]:
def tokenize_text(text):
  if not isinstance(text, str):
    return []

  return text.split()

df['teks_step7'] = df['teks_step6'].apply(tokenize_text)

total_rows = len(df['teks_step7'])
sample_type = type(df['teks_step7'].iloc[0])

print('Tokenization Step 7 Selesai!')
print(f'Total Baris Artikel Terproses: {total_rows}')
print(f'Tipe Data Kolom teks_step7: {sample_type}')

print('\n--- SAMPEL 15 TOKEN PERTAMA (Artikel 1: Algoritma bias) ---')
print(df['teks_step7'].iloc[0][:15])

print('\n--- SAMPEL 15 TOKEN PERTAMA (Artikel 2: Algoritma Lanczos) ---')
print(df['teks_step7'].iloc[1][:15])

Tokenization Step 7 Selesai!
Total Baris Artikel Terproses: 100
Tipe Data Kolom teks_step7: <class 'list'>

--- SAMPEL 15 TOKEN PERTAMA (Artikel 1: Algoritma bias) ---
['algoritma', 'bias', 'menggambarkan', 'galat', 'sistematis', 'dan', 'berulang', 'dalam', 'sistem', 'komputer', 'yang', 'menciptakan', 'hasil', 'tidak', 'adil']

--- SAMPEL 15 TOKEN PERTAMA (Artikel 2: Algoritma Lanczos) ---
['algoritma', 'lanczos', 'adalah', 'algoritma', 'iteratif', 'adaptasi', 'dari', 'metode', 'daya', 'power', 'method', 'untuk', 'menemukan', 'm', 'nilai']


## Step 8 - Stopword Removal Bahasa Indonesia

In [16]:
!pip install Sastrawi

In [17]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

factory = StopWordRemoverFactory()
stopword_list = set(factory.get_stop_words())

def remove_stopwords(tokens):
  if not isinstance(tokens, list):
    return []
  return [token for token in tokens if token not in stopword_list]

df['teks_step8'] = df['teks_step7'].apply(remove_stopwords)

tokens_before = len(df['teks_step7'].iloc[0])
tokens_after = len(df['teks_step8'].iloc[0])
total_stopwords_removed = tokens_before - tokens_after

print('Stopword Removal Step 8 Selesai!')
print(f'Total Kata dalam Kamus Stopword Sastrawi: {len(stopword_list)}')
print(f'\nContoh Evaluasi pada Artikel 1 (Algoritma bias):')
print(f' - Jumlah Token Sebelum Step 8 : {tokens_before} kata')
print(f' - Jumlah Token Setelah Step 8  : {tokens_after} kata')
print(f' - Total Stopword Terbuang     : {total_stopwords_removed} kata')

print('\n15 Token Pertama SEBELUM Stopword Removal:')
print(df['teks_step7'].iloc[0][:15])

print('\n15 Token Pertama SESUDAH Stopword Removal:')
print(df['teks_step8'].iloc[0][:15])

Stopword Removal Step 8 Selesai!
Total Kata dalam Kamus Stopword Sastrawi: 123

Contoh Evaluasi pada Artikel 1 (Algoritma bias):
 - Jumlah Token Sebelum Step 8 : 1281 kata
 - Jumlah Token Setelah Step 8  : 863 kata
 - Total Stopword Terbuang     : 418 kata

15 Token Pertama SEBELUM Stopword Removal:
['algoritma', 'bias', 'menggambarkan', 'galat', 'sistematis', 'dan', 'berulang', 'dalam', 'sistem', 'komputer', 'yang', 'menciptakan', 'hasil', 'tidak', 'adil']

15 Token Pertama SESUDAH Stopword Removal:
['algoritma', 'bias', 'menggambarkan', 'galat', 'sistematis', 'berulang', 'sistem', 'komputer', 'menciptakan', 'hasil', 'adil', 'mengistimewakan', 'satu', 'kategori', 'atas']


## Step 9 - Indonesian Stemming

In [18]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

def stem_tokens(tokens):
  if not isinstance(tokens, list):
    return []

  text = ' '.join(tokens)
  stemmed_text = stemmer.stem(text)

  return stemmed_text.split()

print('Prosedur stemming sedang berjalan (memerlukan waktu ~1-2 menit)...')
df['teks_step9'] = df['teks_step8'].apply(stem_tokens)

print('Stemming Step 9 Selesai!')

print('\n15 Token Pertama SEBELUM Stemming (Step 8):')
print(df['teks_step8'].iloc[0][:15])

print('\n15 Token Pertama SESUDAH Stemming (Step 9):')
print(df['teks_step9'].iloc[0][:15])

Prosedur stemming sedang berjalan (memerlukan waktu ~1-2 menit)...
Stemming Step 9 Selesai!

15 Token Pertama SEBELUM Stemming (Step 8):
['algoritma', 'bias', 'menggambarkan', 'galat', 'sistematis', 'berulang', 'sistem', 'komputer', 'menciptakan', 'hasil', 'adil', 'mengistimewakan', 'satu', 'kategori', 'atas']

15 Token Pertama SESUDAH Stemming (Step 9):
['algoritma', 'bias', 'gambar', 'galat', 'sistematis', 'ulang', 'sistem', 'komputer', 'cipta', 'hasil', 'adil', 'istimewa', 'satu', 'kategori', 'atas']


## Step 10 - Before vs After Analysis

In [19]:
word_count_before = df['teks_step7'].apply(len)
word_count_after = df['teks_step9'].apply(len)

reduction_percentage = (
    (word_count_before.sum() - word_count_after.sum())
    / word_count_before.sum()
) * 100

print('=== ANALISIS EVALUASI PREPROCESSING (BEFORE VS AFTER) ===\n')
print(f'1. Total Kata di Corpus (Sebelum) : {word_count_before.sum():,} kata')
print(f'2. Total Kata di Corpus (Sesudah) : {word_count_after.sum():,} kata')
print(
    '3. Rata-rata Kata per Artikel     :'
    f' {word_count_before.mean():.1f} -> {word_count_after.mean():.1f} kata'
)
print(f'4. Efisiensi Reduksi Noise        : {reduction_percentage:.2f}% kata'
      ' berlebih/derau berhasil dibuang!\n')

print('--- KUALITAS TEKS SEBELUM PREPROCESSING (Raw Text) ---')
print(df[df['judul'] == 'Algoritma Lanczos']['teks'].values[0][:250])

print('\n--- KUALITAS TEKS SESUDAH PREPROCESSING LENGKAP (Clean Tokens) ---')
print(df[df['judul'] == 'Algoritma Lanczos']['teks_step9'].values[0][:30])

=== ANALISIS EVALUASI PREPROCESSING (BEFORE VS AFTER) ===

1. Total Kata di Corpus (Sebelum) : 108,688 kata
2. Total Kata di Corpus (Sesudah) : 81,228 kata
3. Rata-rata Kata per Artikel     : 1086.9 -> 812.3 kata
4. Efisiensi Reduksi Noise        : 25.26% kata berlebih/derau berhasil dibuang!

--- KUALITAS TEKS SEBELUM PREPROCESSING (Raw Text) ---
Algoritma Lanczos adalah algoritma iteratif adaptasi dari metode daya (power method) untuk menemukan 
  
    
      
        m
      
    
    {\displaystyle m}
  
 nilai dan vektor eigen yang "paling berguna" (umumnya yang tertinggi/terendah) dari s

--- KUALITAS TEKS SESUDAH PREPROCESSING LENGKAP (Clean Tokens) ---
['algoritma', 'lanczos', 'algoritma', 'iteratif', 'adaptasi', 'metode', 'daya', 'power', 'method', 'temu', 'm', 'nilai', 'vektor', 'eigen', 'paling', 'guna', 'umum', 'tinggi', 'rendah', 'buah', 'matriks', 'hermite', 'ukur', 'n', 'n', 'm', 'perlu', 'jauh', 'lebih', 'kecil']


## Step 11 - Export Output File CSV

In [20]:
df['teks_clean'] = df['teks_step9'].apply(
    lambda tokens: ' '.join(tokens) if isinstance(tokens, list) else ''
)

df_output = df[['no', 'judul', 'topik', 'teks_clean']].copy()

output_filename = 'hasil_preprocessed.csv'
df_output.to_csv(output_filename, index=False)

print('=== PROSES PREPROCESSING SELESAI 100% ===')
print(f'File berhasil diekspor dengan nama: {output_filename}')
print(f'Dimensi DataFrame Output           : {df_output.shape}')

df_output.head()

=== PROSES PREPROCESSING SELESAI 100% ===
File berhasil diekspor dengan nama: hasil_preprocessed.csv
Dimensi DataFrame Output           : (100, 4)


,no,judul,topik,teks_clean
0,1,Algoritma bias,Computer Science,algoritma bias gambar galat sistematis ulang s...
1,2,Algoritma Lanczos,Computer Science,algoritma lanczos algoritma iteratif adaptasi ...
2,3,ASCII,Computer Science,ascii singkat american standard code for infor...
3,4,Karakter null,Computer Science,karakter null bita null singkat nul null buah ...
4,5,Komputasi optis,Computer Science,komputasi optis guna foton hasil laser maupun ...
